In [ ]:
!pip install boto3 requests requests-aws4auth


In [7]:
!aws sts get-caller-identity


{
    "UserId": "AIDA264MEE3YQ6KVZZNWB",
    "Account": "753523762929",
    "Arn": "arn:aws:iam::753523762929:user/raj3"
}


In [1]:
import boto3
from requests_aws4auth import AWS4Auth
import requests
import json

# Configure
region = "us-east-2"  # your AWS region
collection_endpoint = "https://msoti3kesviob7ogd2i2.us-east-2.aoss.amazonaws.com"  # no trailing slash
index_name = "qna-index"  # your OpenSearch index name

# Setup AWS auth
session = boto3.Session()
credentials = session.get_credentials().get_frozen_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    "aoss",
    session_token=credentials.token
)
headers = { "Content-Type": "application/json" }


In [2]:
# Load Q&A data from your local file
json_file_path = "qanda.json"  # path to your file
with open(json_file_path, "r") as f:
    qna_data = json.load(f)

# Preview data
qna_data[:2]


[{'id': '1',
  'question': 'What is AWS OpenSearch?',
  'answer': 'It is a managed search and analytics engine.'},
 {'id': '2',
  'question': 'What is semantic search?',
  'answer': 'Semantic search understands the meaning behind your query.'}]

In [3]:
index_url = f"{collection_endpoint}/{index_name}"
res = requests.put(index_url, auth=awsauth, headers=headers)
print("Index create status:", res.status_code, res.text)


Index create status: 400 {"error":{"root_cause":[{"type":"resource_already_exists_exception","reason":"OpenSearch exception [type=resource_already_exists_exception, reason=index [qna-index/ygECjJYBCur1WLXB4hwB] already exists]- server : [envoy]"}],"type":"resource_already_exists_exception","reason":"OpenSearch exception [type=resource_already_exists_exception, reason=index [qna-index/ygECjJYBCur1WLXB4hwB] already exists]- server : [envoy]"},"status":400}


In [4]:
# Load Q&A data (again, in case not in memory)
json_file_path = "qanda.json"
with open(json_file_path, "r") as f:
    qna_data = json.load(f)

# Upload each Q&A pair
for i, item in enumerate(qna_data):
    doc_id = item.get("id", str(i))
    doc_url = f"{collection_endpoint}/{index_name}/_doc/{doc_id}"
    res = requests.put(doc_url, auth=awsauth, json=item, headers=headers)
    print(f"Uploading doc {doc_id}:", res.status_code)


Uploading doc 1: 200
Uploading doc 2: 200


In [5]:
search_query = {
    "query": {
        "match": {
            "question": "semantic search"
        }
    }
}

search_url = f"{collection_endpoint}/{index_name}/_search"
res = requests.get(search_url, auth=awsauth, headers=headers, json=search_query)

# Show full response
print(res.status_code)
print(json.dumps(res.json(), indent=2))


200
{
  "took": 1681,
  "timed_out": false,
  "_shards": {
    "total": 0,
    "successful": 0,
    "skipped": 0,
    "failed": 0
  },
  "hits": {
    "total": {
      "value": 1,
      "relation": "eq"
    },
    "max_score": 0.5753642,
    "hits": [
      {
        "_index": "qna-index",
        "_id": "2",
        "_score": 0.5753642,
        "_source": {
          "id": "2",
          "question": "What is semantic search?",
          "answer": "Semantic search understands the meaning behind your query."
        }
      }
    ]
  }
}


In [6]:
search_query = {
    "query": {
        "match": {
            "question": "open search engine"
        }
    }
}


search_url = f"{collection_endpoint}/{index_name}/_search"
res = requests.get(search_url, auth=awsauth, headers=headers, json=search_query)

# Show full response
print(res.status_code)
print(json.dumps(res.json(), indent=2))


# res = requests.get(search_url, auth=awsauth, headers=headers, json=search_query)
# print(json.dumps(res.json(), indent=2))


200
{
  "took": 36,
  "timed_out": false,
  "_shards": {
    "total": 0,
    "successful": 0,
    "skipped": 0,
    "failed": 0
  },
  "hits": {
    "total": {
      "value": 1,
      "relation": "eq"
    },
    "max_score": 0.2876821,
    "hits": [
      {
        "_index": "qna-index",
        "_id": "2",
        "_score": 0.2876821,
        "_source": {
          "id": "2",
          "question": "What is semantic search?",
          "answer": "Semantic search understands the meaning behind your query."
        }
      }
    ]
  }
}


In [7]:
delete_url = f"{collection_endpoint}/{index_name}"
requests.delete(delete_url, auth=awsauth, headers=headers)


<Response [200]>

In [8]:
index_mapping = {
    "settings": {
        "index": {
            "knn": True
        }
    },
    "mappings": {
        "properties": {
            "question": {"type": "text"},
            "answer": {"type": "text"},
            "question_vector": {
                "type": "knn_vector",
                "dimension": 384
            }
        }
    }
}

res = requests.put(
    f"{collection_endpoint}/{index_name}",
    auth=awsauth,
    headers=headers,
    json=index_mapping
)
print("Index recreate status:", res.status_code)


Index recreate status: 400


In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

with open("qanda.json", "r") as f:
    qna_data = json.load(f)

for i, item in enumerate(qna_data):
    question_text = item["question"]
    vector = model.encode(question_text).tolist()
    item["question_vector"] = vector

    doc_id = item.get("id", str(i))
    doc_url = f"{collection_endpoint}/{index_name}/_doc/{doc_id}"
    res = requests.put(doc_url, auth=awsauth, json=item, headers=headers)
    print(f"Indexed doc {doc_id}:", res.status_code)


Indexed doc 1: 201
Indexed doc 2: 201


In [12]:
query_text = "What is a search engine that understands meaning?"
query_vector = model.encode(query_text).tolist()

semantic_query = {
    "size": 2,
    "query": {
        "knn": {
            "question_vector": {
                "vector": query_vector,
                "k": 2
            }
        }
    }
}

res = requests.get(
    f"{collection_endpoint}/{index_name}/_search",
    auth=awsauth,
    headers=headers,
    json=semantic_query
)

print("Semantic Search Results:")
print(res.status_code)
print(json.dumps(res.json(), indent=2))

# for hit in res.json()["hits"]["hits"]:
#     print(f"→ {hit['_source']['question']}\n  {hit['_source']['answer']}\n")


Semantic Search Results:
400
{
  "error": {
    "root_cause": [
      {
        "type": "query_shard_exception",
        "reason": "Unsupported query type",
        "index": "753523762929::msoti3kesviob7ogd2i2::SEARCH::qna-index:0",
        "index_uuid": "sNUZjJYBdbA3xBFOyPcV:0"
      }
    ],
    "type": "search_phase_execution_exception",
    "reason": "all shards failed",
    "phase": "query",
    "grouped": true,
    "failed_shards": [
      {
        "shard": 0,
        "index": "qna-index",
        "node": "---",
        "reason": {
          "type": "query_shard_exception",
          "reason": "Unsupported query type",
          "index": "753523762929::msoti3kesviob7ogd2i2::SEARCH::qna-index:0",
          "index_uuid": "sNUZjJYBdbA3xBFOyPcV:0"
        }
      }
    ]
  },
  "status": 400
}


In [13]:
query_text = "What is a search engine that understands meaning?"
query_vector = model.encode(query_text).tolist()

semantic_query = {
    "size": 2,
    "query": {
        "knn": {
            "question_vector": {
                "vector": query_vector,
                "k": 2
            }
        }
    }
}

res = requests.post(  # <-- POST instead of GET
    f"{collection_endpoint}/{index_name}/_search",
    auth=awsauth,
    headers=headers,
    json=semantic_query
)

# View entire response for debugging
print(json.dumps(res.json(), indent=2))

# Optional: print nicely if 'hits' are present
hits = res.json().get("hits", {}).get("hits", [])
if hits:
    print("\nSemantic Search Results:")
    for hit in hits:
        print(f"→ {hit['_source']['question']}\n  {hit['_source']['answer']}\n")
else:
    print("⚠️ No results returned.")


{
  "error": {
    "root_cause": [
      {
        "type": "query_shard_exception",
        "reason": "Unsupported query type",
        "index": "753523762929::msoti3kesviob7ogd2i2::SEARCH::qna-index:0",
        "index_uuid": "sNUZjJYBdbA3xBFOyPcV:0"
      }
    ],
    "type": "search_phase_execution_exception",
    "reason": "all shards failed",
    "phase": "query",
    "grouped": true,
    "failed_shards": [
      {
        "shard": 0,
        "index": "qna-index",
        "node": "---",
        "reason": {
          "type": "query_shard_exception",
          "reason": "Unsupported query type",
          "index": "753523762929::msoti3kesviob7ogd2i2::SEARCH::qna-index:0",
          "index_uuid": "sNUZjJYBdbA3xBFOyPcV:0"
        }
      }
    ]
  },
  "status": 400
}
⚠️ No results returned.


In [15]:
# Delete index if it already exists (ignore error if not found)
requests.delete(f"{collection_endpoint}/{index_name}", auth=awsauth, headers=headers)

# Create index with knn_vector
index_mapping = {
    "settings": {
        "index": {
            "knn": True
        }
    },
    "mappings": {
        "properties": {
            "question": { "type": "text" },
            "answer": { "type": "text" },
            "question_vector": {
                "type": "knn_vector",
                "dimension": 384
            }
        }
    }
}

res = requests.put(f"{collection_endpoint}/{index_name}", auth=awsauth, headers=headers, json=index_mapping)
print("Index creation status:", res.status_code)


Index creation status: 400


In [8]:
# https://zk4wmofkl4vfvow46uad.us-east-1.aoss.amazonaws.com


# Replace this with your actual endpoint for the vector collection (no trailing slash)
collection_endpoint = "https://zk4wmofkl4vfvow46uad.us-east-1.aoss.amazonaws.com"
index_name = "rajtest-index"  # your OpenSearch index name

headers = {"Content-Type": "application/json"}

# Keep your AWS credentials via boto3
import boto3
from requests_aws4auth import AWS4Auth
import requests
import json

session = boto3.Session()
credentials = session.get_credentials().get_frozen_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    session.region_name or "us-east-1",
    "aoss",
    session_token=credentials.token
)


In [9]:
# Delete index if it already exists (ignore error if not found)
requests.delete(f"{collection_endpoint}/{index_name}", auth=awsauth, headers=headers)

# Create index with knn_vector
index_mapping = {
    "settings": {
        "index": {
            "knn": True
        }
    },
    "mappings": {
        "properties": {
            "question": { "type": "text" },
            "answer": { "type": "text" },
            "question_vector": {
                "type": "knn_vector",
                "dimension": 384
            }
        }
    }
}

res = requests.put(f"{collection_endpoint}/{index_name}", auth=awsauth, headers=headers, json=index_mapping)
print("Index creation status:", res.status_code)


Index creation status: 403


In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

with open("qanda.json", "r") as f:
    qna_data = json.load(f)

for i, item in enumerate(qna_data):
    vector = model.encode(item["question"]).tolist()
    item["question_vector"] = vector
    doc_id = item.get("id", str(i))
    res = requests.put(
        f"{collection_endpoint}/{index_name}/_doc/{doc_id}",
        auth=awsauth,
        headers=headers,
        json=item
    )
    print(f"Uploaded doc {doc_id}:", res.status_code)


Uploaded doc 1: 403
Uploaded doc 2: 403


In [6]:
query_text = "Search engine that understands meaning"
query_vector = model.encode(query_text).tolist()

semantic_query = {
    "size": 2,
    "query": {
        "knn": {
            "question_vector": {
                "vector": query_vector,
                "k": 2
            }
        }
    }
}

res = requests.post(
    f"{collection_endpoint}/{index_name}/_search",
    auth=awsauth,
    headers=headers,
    json=semantic_query
)

print("Semantic Search Results:\n")
for hit in res.json().get("hits", {}).get("hits", []):
    print(f"→ {hit['_source']['question']}\n  {hit['_source']['answer']}\n")


Semantic Search Results:

